In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
print("Environment ready!")
print("Pandas version:", pd.__version__)

In [ ]:
# Check possible dataset locations
candidates = [
    Path("data/sales.csv"),
    Path("backend/data/sales.csv")
]

DATA_PATH = next(
    (path for path in candidates if path.exists()),
    None
)

print("Current directory:", Path.cwd())
print("Dataset found:", DATA_PATH)

In [ ]:
# Load CSV into a Pandas DataFrame
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate transaction IDs:", df["Transaction_Id"].duplicated().sum())

In [ ]:
df.describe(include="all")

In [ ]:
for col in df.select_dtypes(include="object").columns:
    print(f"\n{col}:")
    print(df[col].unique())

In [ ]:
df["Status"].value_counts()

In [ ]:
df[df.isnull().any(axis=1)]

In [ ]:
df[df["Quantity"] <= 0]

In [ ]:
df[df["Unit_price"] <= 0]

In [ ]:
df[df["Discount_amount"] < 0]

In [ ]:
df[
    df["Discount_amount"] >
    (df["Quantity"] * df["Unit_price"])
]

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

print(
    "Duplicate transaction IDs:",
    df["Transaction_Id"].duplicated().sum()
)

In [ ]:
df["Order_date"] = pd.to_datetime(
    df["Order_date"],
    format="%d-%m-%Y",
    errors="coerce"
)

print("Invalid dates:", df["Order_date"].isna().sum())

df[["Transaction_Id", "Order_date"]].head()

In [ ]:
df.select_dtypes(include=["object", "str"])

## Create a working copy

In [ ]:
clean_df = df.copy()

print("Working copy created!")
print("Original records:", len(df))

## Standardize column names and text

In [ ]:
# Remove spaces around column names
clean_df.columns = clean_df.columns.str.strip()

# Remove extra spaces in text values
text_columns = [
    "Transaction_Id",
    "Product",
    "Category",
    "Region",
    "Sales_channel",
    "Status"
]

for col in text_columns:
    clean_df[col] = clean_df[col].str.strip()

print("Text cleaning completed!")

## Convert data types

In [ ]:
clean_df["Order_date"] = pd.to_datetime(
    clean_df["Order_date"],
    errors="coerce"
)

numeric_columns = [
    "Quantity",
    "Unit_price",
    "Discount_amount"
]

for col in numeric_columns:
    clean_df[col] = pd.to_numeric(
        clean_df[col],
        errors="coerce"
    )

print(clean_df.dtypes)

## Identify rejected transactions

In [ ]:
reasons = pd.Series(
    "",
    index=clean_df.index,
    dtype="object"
)

def add_reason(condition, message):
    reasons.loc[condition] += message + "; "

# Missing required values
required_columns = [
    "Transaction_Id",
    "Order_date",
    "Product",
    "Category",
    "Region",
    "Sales_channel",
    "Quantity",
    "Unit_price",
    "Discount_amount",
    "Status"
]

for col in required_columns:
    add_reason(
        clean_df[col].isna(),
        f"Missing {col}"
    )

# Invalid quantity
add_reason(
    clean_df["Quantity"] <= 0,
    "Invalid quantity"
)

# Invalid unit price
add_reason(
    clean_df["Unit_price"] <= 0,
    "Invalid unit price"
)

# Negative discount
add_reason(
    clean_df["Discount_amount"] < 0,
    "Negative discount"
)

# Discount exceeding gross amount
gross = clean_df["Quantity"] * clean_df["Unit_price"]

add_reason(
    clean_df["Discount_amount"] > gross,
    "Discount exceeds gross amount"
)

# Returned or non-completed transactions
add_reason(
    clean_df["Status"].str.lower() != "completed",
    "Transaction not completed"
)

# Separate datasets
valid_mask = reasons.eq("")

cleaned_df = clean_df.loc[valid_mask].copy()

rejected_df = clean_df.loc[~valid_mask].copy()

rejected_df["Rejection_Reason"] = (
    reasons.loc[~valid_mask].str.rstrip("; ")
)

print("Valid records:", len(cleaned_df))
print("Rejected records:", len(rejected_df))

## Calculate revenue

In [ ]:
cleaned_df["Revenue"] = (
    cleaned_df["Quantity"] * cleaned_df["Unit_price"]
    - cleaned_df["Discount_amount"]
)

cleaned_df.head()

## Create monthly feature

In [ ]:
cleaned_df["Order_Month"] = (
    cleaned_df["Order_date"].dt.to_period("M").astype(str)
)

cleaned_df.head()

## Save both datasets

In [ ]:
cleaned_df.to_csv(
    "data/cleaned_sales.csv",
    index=False
)

rejected_df.to_csv(
    "data/rejected_sales.csv",
    index=False
)

print("Datasets saved successfully!")

## Final validation

In [ ]:
print("Original:", len(df))
print("Cleaned:", len(cleaned_df))
print("Rejected:", len(rejected_df))

print("\nMissing values:")
print(cleaned_df.isnull().sum())

print("\nTotal revenue:")
print(cleaned_df["Revenue"].sum())

In [ ]:
print("Cleaned dataset shape:", cleaned_df.shape)

print("\nMissing values:")
print(cleaned_df.isnull().sum())

print("\nTotal revenue:")
print(cleaned_df["Revenue"].sum())

print("\nRejected transaction details:")
print(
    rejected_df[
        ["Transaction_Id", "Rejection_Reason"]
    ]
)